<a href="https://colab.research.google.com/github/arasuezhile/stkProj/blob/dev/Weekly_Stock_Forecaster_Comparison_with_Previous_week.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# main.py
# In a Google Colab or similar environment, run the following command in a separate
# cell before executing the rest of the script to install the required libraries.
#!pip install --upgrade yfinance pandas "numpy<2.0" pandas_ta scikit-learn "curl_cffi"

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import warnings
import json
import os
from datetime import datetime, timedelta
import time
import requests

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)

PREDICTIONS_FILE = 'predictions_log.json'

def get_stock_data(ticker_symbol, period="max"):
    """
    Fetches historical price data using the yf.Ticker method.
    """
    try:
        stock = yf.Ticker(ticker_symbol)
        hist_data = stock.history(period=period, auto_adjust=True)
        if hist_data.empty:
            print(f"Error: No historical data found for ticker '{ticker_symbol}'. It might be delisted or an invalid ticker.")
            return None
        return hist_data
    except Exception as e:
        print(f"An error occurred while fetching data for {ticker_symbol}: {e}")
        return None

def create_weekly_technical_features(daily_df):
    """
    Resamples daily data to weekly and calculates a standard set of technical indicators.
    """
    if daily_df is None:
        return None

    # Resample daily data to weekly data (Monday-based weeks)
    weekly_df = daily_df.resample('W-MON').agg({
        'Open': 'first', 'High': 'max', 'Low': 'min',
        'Close': 'last', 'Volume': 'sum'
    }).dropna()

    # Calculate a standard set of technical indicators on the weekly data
    weekly_df.ta.sma(length=10, append=True)
    weekly_df.ta.sma(length=20, append=True)
    weekly_df.ta.rsi(length=14, append=True)
    weekly_df.ta.macd(length=8, fast=12, slow=26, append=True)
    weekly_df.ta.bbands(length=20, append=True)
    weekly_df.ta.obv(append=True)

    # Rename columns for consistency
    weekly_df.rename(columns={
        'SMA_10': 'sma_10', 'SMA_20': 'sma_20', 'RSI_14': 'rsi_14',
        'MACD_12_26_8': 'macd', 'MACDh_12_26_8': 'macd_h', 'MACDs_12_26_8': 'macd_s',
        'BBL_20_2.0': 'bb_low', 'BBM_20_2.0': 'bb_mid', 'BBU_20_2.0': 'bb_high',
        'BBB_20_2.0': 'bb_band', 'BBP_20_2.0': 'bb_percent', 'OBV': 'obv'
    }, inplace=True)

    weekly_df.dropna(inplace=True)
    return weekly_df

def train_and_predict(training_data, forecast_data, features):
    """
    A helper function to train a model and make a single prediction.
    """
    if len(training_data) < 1:
        return None, None

    X_train = training_data[features]
    y_train = training_data['target']

    model = RandomForestRegressor(n_estimators=100, random_state=42, min_samples_split=10)
    model.fit(X_train, y_train)

    X_forecast = forecast_data[features]
    predicted_change = model.predict(X_forecast)[0]

    feature_importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

    return predicted_change, feature_importances

def load_previous_predictions():
    """
    Loads saved predictions from the log file.
    """
    if os.path.exists(PREDICTIONS_FILE):
        with open(PREDICTIONS_FILE, 'r') as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return {} # Return empty dict if file is corrupted or empty
    return {}

def save_new_predictions(predictions):
    """
    Saves the new set of predictions to the log file.
    """
    with open(PREDICTIONS_FILE, 'w') as f:
        json.dump(predictions, f, indent=4)
    print(f"\nSuccessfully saved new forecasts to {PREDICTIONS_FILE}")

def format_notification_message(predictions):
    """
    Formats the prediction data into a single string suitable for a notification.
    """
    message_lines = [f"--- Weekly Stock Forecasts: {datetime.now().strftime('%Y-%m-%d')} ---"]
    for ticker, data in predictions.items():
        direction = "UP" if data['predicted_change'] > 0 else "DOWN"
        line = f"{ticker}: {direction} by {data['predicted_change']:.2f}% (Target: ₹{data['predicted_price']:.2f})"
        message_lines.append(line)
    return "\n".join(message_lines)

def send_notification(message):
    """
    Placeholder function for sending a notification.
    In a real application, you would integrate a notification service here.
    """
    print("\n\n================ NOTIFICATION PREVIEW ================")
    print(message)
    print("==================================================")
    print("(In a real system, this message would be sent to your phone via a service like Twilio, Pushbullet, or another webhook.)")


def run_weekly_automation(ticker_list):
    """
    Orchestrates the weekly validation and forecasting process with robust error handling.
    """
    print(f"--- Weekly Stock Forecast & Validation ---")
    print(f"--- Date: {datetime.now().strftime('%Y-%m-%d %H:%M')} ---")

    previous_predictions = load_previous_predictions()
    new_predictions = {}

    for ticker in ticker_list:
        time.sleep(1)
        print(f"\n\n{'='*40}")
        print(f"  Processing: {ticker}")
        print(f"{'='*40}")

        try:
            daily_data = get_stock_data(ticker)
            if daily_data is None:
                continue

            # --- 1. Validation of Last Week's Forecast ---
            if ticker in previous_predictions:
                pred_info = previous_predictions[ticker]
                last_pred_date = datetime.strptime(pred_info['forecast_for_week_of'], '%Y-%m-%d').date()

                if not daily_data.empty and daily_data.index.max().date() >= last_pred_date + timedelta(days=4):
                    try:
                        validation_date = pd.to_datetime(str(last_pred_date + timedelta(days=4)))
                        actual_close = daily_data.asof(validation_date)['Close']

                        predicted_close = pred_info['predicted_price']
                        change = ((actual_close - pred_info['price_at_forecast']) / pred_info['price_at_forecast']) * 100

                        print("\n--- Validation Report for Last Week ---")
                        print(f"Forecast was for week of: {last_pred_date}")
                        print(f"Predicted Target: ₹{predicted_close:.2f} ({pred_info['predicted_change']:.2f}%)")
                        print(f"Actual Closing Price: ₹{actual_close:.2f} ({change:.2f}%)")

                        pred_dir = "UP" if pred_info['predicted_change'] > 0 else "DOWN"
                        actual_dir = "UP" if change > 0 else "DOWN"

                        if pred_dir == actual_dir:
                            print("Result: CORRECT Direction")
                        else:
                            print("Result: INCORRECT Direction")

                    except (KeyError, IndexError):
                        print(f"\nCould not find actual data near {last_pred_date + timedelta(days=4)} to validate forecast.")
                else:
                     print("\nLast week's data not yet available for validation.")

            # --- 2. Generate New Forecast for Upcoming Week ---
            weekly_features = create_weekly_technical_features(daily_data)
            if weekly_features is None:
                print("Not enough data to generate new forecast.")
                continue

            # This is a temporary structure for a 1-week prediction model
            temp_data = weekly_features.copy()
            temp_data['target'] = (temp_data['Close'].shift(-1) - temp_data['Close']) / temp_data['Close'] * 100

            training_data = temp_data.iloc[:-1].dropna(subset=['target'])
            forecast_input_data = temp_data.iloc[-1:].copy()

            features = [col for col in temp_data.columns if col not in ['Open', 'High', 'Low', 'Close', 'Volume', 'target']]

            predicted_change, importances = train_and_predict(training_data, forecast_input_data, features)

            if predicted_change is not None:
                last_close_price = weekly_features.iloc[-1]['Close']
                predicted_price = last_close_price * (1 + predicted_change / 100)

                forecast_date = weekly_features.index[-1]
                start_of_forecast_period = pd.to_datetime(forecast_date).date() + timedelta(days=7)
                direction = "UP" if predicted_change > 0 else "DOWN"

                print("\n--- New Forecast for Upcoming Week ---")
                print(f"Prediction for week starting {start_of_forecast_period}: {direction} by {predicted_change:.2f}%")

                new_predictions[ticker] = {
                    'forecast_for_week_of': start_of_forecast_period.strftime('%Y-%m-%d'),
                    'price_at_forecast': last_close_price,
                    'predicted_change': predicted_change,
                    'predicted_price': predicted_price
                }

        except Exception as e:
            print(f"\n### AN UNEXPECTED ERROR OCCURRED FOR {ticker} ###")
            print(f"Error details: {e}")
            print("Skipping this ticker and moving to the next one.")
            continue

    # Save all the new forecasts and generate the notification
    if new_predictions:
        save_new_predictions(new_predictions)
        notification_message = format_notification_message(new_predictions)
        send_notification(notification_message)


if __name__ == "__main__":
    # --- USER INPUT ---
    # Add the list of NSE stock tickers you want to track here.
    # Note: MAANALU.NS is an example of an INCORRECT ticker to show error handling.
    ticker_list = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "HINDZINC.NS", "TATASTEEL.NS"]

    run_weekly_automation(ticker_list)


--- Weekly Stock Forecast & Validation ---
--- Date: 2025-06-15 02:50 ---


  Processing: RELIANCE.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: UP by 0.38%


  Processing: TCS.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: DOWN by -0.23%


  Processing: INFY.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: DOWN by -0.88%


  Processing: HDFCBANK.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: UP by 0.76%


ERROR:yfinance:$ZOMATO.NS: possibly delisted; no timezone found




  Processing: ZOMATO.NS
Error: No historical data found for ticker 'ZOMATO.NS'. It might be delisted or an invalid ticker.


  Processing: MANALIPETC.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: DOWN by -1.80%


ERROR:yfinance:$ELECTROCAS.NS: possibly delisted; no timezone found




  Processing: ELECTROCAS.NS
Error: No historical data found for ticker 'ELECTROCAS.NS'. It might be delisted or an invalid ticker.


  Processing: HINDZINC.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: UP by 6.15%


  Processing: TATASTEEL.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: UP by 0.09%


  Processing: NATIONALUM.NS

--- New Forecast for Upcoming Week ---
Prediction for week starting 2025-06-23: DOWN by -0.48%

Successfully saved new forecasts to predictions_log.json


================ NOTIFICATION PREVIEW ================
--- Weekly Stock Forecasts: 2025-06-15 ---
RELIANCE.NS: UP by 0.38% (Target: ₹1433.30)
TCS.NS: DOWN by -0.23% (Target: ₹3437.64)
INFY.NS: DOWN by -0.88% (Target: ₹1587.93)
HDFCBANK.NS: UP by 0.76% (Target: ₹1932.25)
MANALIPETC.NS: DOWN by -1.80% (Target: ₹62.85)
HINDZINC.NS: UP by 6.15% (Target: ₹545.89)
TATASTEEL.NS: UP by 0.09% (Target: ₹152.27)
NATIONALUM.NS: DOWN by -0.48% (Ta